# RQ4 — Temporal SATD Early Warning

## Research question

**How early during a sprint can accumulated planning, scope, estimation, workflow, and ownership signals provide a reliable and practically useful warning of subsequent SATD?**

## Answer

**No evaluated checkpoint provides a reliable and practically useful warning under the predeclared operational rule.** The aligned panel contains **9,507 issues from 25 projects**, including **2,128 eventual SATD labels (22.38%)**, observed at T0, T25, T50, and T75. Out-of-project XGBoost PR-AUC changes only from **0.290 at T0** to **0.288 at T25**, **0.291 at T50**, and **0.295 at T75**; corresponding ROC-AUC values are **0.582, 0.581, 0.585, and 0.587**. Thus, accumulating in-sprint signals adds very little discrimination beyond planning-time information. The T0→T25 PR-AUC change is **−0.0019** (95% project-bootstrap CI: −0.0069 to 0.0031), T25→T50 is **+0.0037** (−0.0012 to 0.0080), and T50→T75 is **+0.0049** (0.0000 to 0.0107). Only the final incremental improvement is clearly positive, and it is very small.

At the operational **20% review budget**, recall is only **26.56% at T0**, **26.68% at T25**, **27.28% at T50**, and **27.26% at T75**, while lift is **1.33–1.36×**. Every checkpoint therefore fails the predeclared requirement of recall **at least 50%** and lift **at least 2.0×**. Consequently, there is **no defensible earliest warning point**: T0 already contains nearly all available predictive information, and waiting until T75 yields only a marginal improvement while leaving 25% of the sprint for intervention. The outcome is an eventual SATD label derived from final/current issue text; because the data do not identify when SATD first appeared, these results cannot establish that any warning precedes SATD formation.


## Reproducibility and decision rules

- Input: `rq4_temporal_dataset.csv` when present, otherwise
  `rq4_temporal_long_dataset.csv`.
- Outer validation: five-fold `StratifiedGroupKFold`, grouped by project.
- Alert threshold: selected inside each outer training fold by inner grouped CV;
  the held-out fold never selects its own threshold.
- Operational rule (declared before inspecting model results): at a 20% review
  budget, mean recall must be at least 50% and mean lift at least 2.0.
- Uncertainty: projects, rather than individual issues, are resampled.
- Random seed: 42.

Required packages: `pandas`, `numpy`, `scikit-learn`, `xgboost`, and `matplotlib`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
CHECKPOINT_ORDER = ["T0", "T25", "T50", "T75"]
REMAINING_FRACTION = {"T0": 1.00, "T25": 0.75, "T50": 0.50, "T75": 0.25}
ALERT_BUDGETS = [0.10, 0.20, 0.30]
ACTIONABLE_BUDGET = 0.20
MIN_RECALL = 0.50
MIN_LIFT = 2.00
N_BOOT = 1000
N_PAIRED_BOOT = 2000
RUN_PERMUTATION_IMPORTANCE = True

OUTPUT_DIR = Path("rq4_results")
OUTPUT_DIR.mkdir(exist_ok=True)
candidates = [Path("rq4_temporal_dataset.csv"), Path("rq4_temporal_long_dataset.csv")]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"None of the RQ4 input files exists: {candidates}")

print(f"Input: {DATA_PATH.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")


Input: E:\SATD-Agile\rq4_temporal_long_dataset.csv
Output: E:\SATD-Agile\rq4_results


## 1. Load, clean, and audit the temporal panel

The primary comparison is restricted to issues with one row at every checkpoint.
Label and project identity must remain constant within an issue. Audit tables are
written even when no violations are found.


In [2]:
raw_df = pd.read_csv(DATA_PATH)
required_columns = {"issue_id", "project_id", "sprint_id", "checkpoint", "is_satd"}
missing_required = sorted(required_columns - set(raw_df.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df = raw_df.copy()
df["is_satd"] = pd.to_numeric(df["is_satd"], errors="coerce")
df = df[df["is_satd"].isin([0, 1]) & df["checkpoint"].isin(CHECKPOINT_ORDER)].copy()
df["is_satd"] = df["is_satd"].astype(int)
df["checkpoint"] = pd.Categorical(
    df["checkpoint"], categories=CHECKPOINT_ORDER, ordered=True
)

duplicate_keys = df.duplicated(["issue_id", "checkpoint"], keep=False)
duplicate_rows = df.loc[duplicate_keys].sort_values(["issue_id", "checkpoint"])
duplicate_rows.to_csv(OUTPUT_DIR / "duplicate_issue_checkpoint_rows.csv", index=False)
if duplicate_keys.any():
    raise ValueError(
        f"Found {duplicate_keys.sum()} rows with duplicate issue/checkpoint keys; "
        "resolve them before modelling."
    )

checkpoint_counts = df.groupby("issue_id", observed=True)["checkpoint"].nunique()
incomplete = checkpoint_counts[checkpoint_counts != len(CHECKPOINT_ORDER)]
incomplete.rename("checkpoint_count").to_csv(OUTPUT_DIR / "incomplete_issues.csv")
complete_issue_ids = checkpoint_counts[checkpoint_counts == len(CHECKPOINT_ORDER)].index
df = df[df["issue_id"].isin(complete_issue_ids)].copy()

label_nunique = df.groupby("issue_id")["is_satd"].nunique()
project_nunique = df.groupby("issue_id")["project_id"].nunique()
inconsistent_labels = label_nunique[label_nunique > 1]
inconsistent_projects = project_nunique[project_nunique > 1]
inconsistent_labels.rename("label_count").to_csv(OUTPUT_DIR / "inconsistent_issue_labels.csv")
inconsistent_projects.rename("project_count").to_csv(OUTPUT_DIR / "inconsistent_issue_projects.csv")
if len(inconsistent_labels) or len(inconsistent_projects):
    raise ValueError(
        f"Panel integrity failed: {len(inconsistent_labels)} inconsistent labels and "
        f"{len(inconsistent_projects)} inconsistent project assignments."
    )

base_df = df[df["checkpoint"] == "T0"].copy()
satd_prevalence = base_df["is_satd"].mean()
panel_audit = pd.DataFrame([{
    "raw_rows": len(raw_df),
    "analysis_rows": len(df),
    "complete_issues": df["issue_id"].nunique(),
    "excluded_incomplete_issues": len(incomplete),
    "projects": df["project_id"].nunique(),
    "sprints": df[["project_id", "sprint_id"]].drop_duplicates().shape[0],
    "satd_issues": int(base_df["is_satd"].sum()),
    "satd_prevalence": satd_prevalence,
}])
panel_audit.to_csv(OUTPUT_DIR / "panel_audit.csv", index=False)
print(panel_audit.to_string(index=False))
print("\nRows by checkpoint:")
print(df["checkpoint"].value_counts().sort_index().to_string())


 raw_rows  analysis_rows  complete_issues  excluded_incomplete_issues  projects  sprints  satd_issues  satd_prevalence
    38028          38028             9507                           0        25     1732         2128         0.223835

Rows by checkpoint:
checkpoint
T0     9507
T25    9507
T50    9507
T75    9507


In [3]:
cumulative_candidates = [
    "all_change_count", "description_change_count", "summary_change_count",
    "acceptance_criteria_change_count", "issue_type_change_count",
    "scope_instability_count", "story_point_change_count",
    "actual_story_point_change_count", "time_estimate_change_count",
    "estimation_instability_count", "assignee_change_count", "team_change_count",
    "sprint_change_count", "priority_change_count", "ownership_instability_count",
    "status_transition_count", "blocked_count", "reopen_count",
    "review_return_count", "testing_return_count", "rework_entry_count",
    "backward_transition_count", "comment_count", "unique_commenters",
    "unique_change_authors",
]
cumulative_features = [name for name in cumulative_candidates if name in df.columns]

accumulation_rows = []
t0_rows = []
for feature in cumulative_features:
    numeric = pd.to_numeric(df[feature], errors="coerce")
    wide = (
        df.assign(_value=numeric)
          .pivot(index="issue_id", columns="checkpoint", values="_value")
          .reindex(columns=CHECKPOINT_ORDER)
    )
    adjacent_violations = wide.diff(axis=1).iloc[:, 1:].lt(0).any(axis=1)
    accumulation_rows.append({
        "feature": feature,
        "violating_issues": int(adjacent_violations.sum()),
        "negative_observations": int((numeric < 0).sum()),
    })
    t0_values = wide["T0"]
    t0_rows.append({
        "feature": feature,
        "nonzero_t0_issues": int(t0_values.fillna(0).ne(0).sum()),
        "max_t0_value": t0_values.max(),
    })

accumulation_audit = pd.DataFrame(accumulation_rows)
t0_dynamic_audit = pd.DataFrame(t0_rows)
accumulation_audit.to_csv(OUTPUT_DIR / "temporal_accumulation_check.csv", index=False)
t0_dynamic_audit.to_csv(OUTPUT_DIR / "t0_dynamic_signal_check.csv", index=False)
print("Accumulation violations by feature (non-zero only):")
print(accumulation_audit.query("violating_issues > 0 or negative_observations > 0").to_string(index=False))
print("\nNon-zero T0 signals (non-zero only):")
print(t0_dynamic_audit.query("nonzero_t0_issues > 0").to_string(index=False))


Accumulation violations by feature (non-zero only):
Empty DataFrame
Columns: [feature, violating_issues, negative_observations]
Index: []

Non-zero T0 signals (non-zero only):
Empty DataFrame
Columns: [feature, nonzero_t0_issues, max_t0_value]
Index: []


## 2. Leakage-safe feature definitions

Planning variables are fixed at sprint start. Dynamic event counts are cumulative
only through the checkpoint. Aggregate process counts are deliberately excluded
from the model when their component counts are present, avoiding redundant double
representation. Count features use `log1p`.


In [4]:
sprint_columns = [
    "project_id", "sprint_id", "sprint_issue_count", "estimated_issue_count",
    "sprint_total_story_points",
]
missing_sprint_columns = [name for name in sprint_columns if name not in base_df.columns]
if missing_sprint_columns:
    raise ValueError(f"Missing sprint normalization fields: {missing_sprint_columns}")

sprint_consistency = (
    base_df.groupby(["project_id", "sprint_id"])[
        ["sprint_issue_count", "estimated_issue_count", "sprint_total_story_points"]
    ].nunique(dropna=False).max(axis=1)
)
inconsistent_sprints = sprint_consistency[sprint_consistency > 1]
inconsistent_sprints.rename("max_distinct_values").to_csv(
    OUTPUT_DIR / "inconsistent_sprint_aggregates.csv"
)
if len(inconsistent_sprints):
    raise ValueError(f"Found {len(inconsistent_sprints)} sprints with inconsistent aggregates.")

sprint_base = base_df[sprint_columns].drop_duplicates(["project_id", "sprint_id"]).copy()
for name in ["sprint_issue_count", "estimated_issue_count", "sprint_total_story_points"]:
    sprint_base[name] = pd.to_numeric(sprint_base[name], errors="coerce")
sprint_base["sprint_issue_count_percentile"] = (
    sprint_base.groupby("project_id")["sprint_issue_count"].rank(pct=True)
)
sprint_base["sprint_sp_load_percentile"] = (
    sprint_base.groupby("project_id")["sprint_total_story_points"].rank(pct=True)
)
sprint_base["estimated_issue_ratio"] = (
    sprint_base["estimated_issue_count"] /
    sprint_base["sprint_issue_count"].replace(0, np.nan)
)
df = df.merge(
    sprint_base[[
        "project_id", "sprint_id", "sprint_issue_count_percentile",
        "sprint_sp_load_percentile", "estimated_issue_ratio",
    ]],
    on=["project_id", "sprint_id"], how="left", validate="many_to_one",
)
df["log_issue_age"] = np.log1p(
    pd.to_numeric(df["issue_age_days_at_sprint_start"], errors="coerce").clip(lower=0)
)

planning_categorical = ["priority_group", "issue_type_group"]
planning_numeric = [
    "project_sp_percentile", "story_point_missing_or_zero", "log_issue_age",
    "sprint_issue_count_percentile", "sprint_sp_load_percentile",
    "estimated_issue_ratio", "urgent_issue_ratio",
]
dynamic_families_raw = {
    "Scope": [
        "description_change_count", "summary_change_count",
        "acceptance_criteria_change_count", "issue_type_change_count",
    ],
    "Estimation": [
        "story_point_change_count", "actual_story_point_change_count",
        "time_estimate_change_count",
    ],
    "Ownership": [
        "assignee_change_count", "team_change_count", "sprint_change_count",
        "priority_change_count",
    ],
    "Workflow": [
        "status_transition_count", "blocked_count", "reopen_count",
        "review_return_count", "testing_return_count", "rework_entry_count",
        "backward_transition_count",
    ],
    "Communication": ["comment_count", "unique_commenters", "unique_change_authors"],
}
dynamic_raw = [
    feature for features in dynamic_families_raw.values() for feature in features
    if feature in df.columns
]
for feature in dynamic_raw:
    values = pd.to_numeric(df[feature], errors="coerce").fillna(0).clip(lower=0)
    df[feature] = values
    df[f"log_{feature}"] = np.log1p(values)
dynamic_features = [f"log_{feature}" for feature in dynamic_raw]
all_features = planning_categorical + planning_numeric + dynamic_features

missing_model_features = sorted(set(planning_categorical + planning_numeric) - set(df.columns))
if missing_model_features:
    raise ValueError(f"Missing model features: {missing_model_features}")
print(f"Model features: {len(all_features)} ({len(planning_categorical)} categorical, "
      f"{len(planning_numeric) + len(dynamic_features)} numeric)")
print(json.dumps(all_features, indent=2))


Model features: 30 (2 categorical, 28 numeric)
[
  "priority_group",
  "issue_type_group",
  "project_sp_percentile",
  "story_point_missing_or_zero",
  "log_issue_age",
  "sprint_issue_count_percentile",
  "sprint_sp_load_percentile",
  "estimated_issue_ratio",
  "urgent_issue_ratio",
  "log_description_change_count",
  "log_summary_change_count",
  "log_acceptance_criteria_change_count",
  "log_issue_type_change_count",
  "log_story_point_change_count",
  "log_actual_story_point_change_count",
  "log_time_estimate_change_count",
  "log_assignee_change_count",
  "log_team_change_count",
  "log_sprint_change_count",
  "log_priority_change_count",
  "log_status_transition_count",
  "log_blocked_count",
  "log_reopen_count",
  "log_review_return_count",
  "log_testing_return_count",
  "log_rework_entry_count",
  "log_backward_transition_count",
  "log_comment_count",
  "log_unique_commenters",
  "log_unique_change_authors"
]


In [5]:
signal_family_columns = [
    name for name in [
        "scope_instability_count", "estimation_instability_count",
        "ownership_instability_count", "status_transition_count",
    ] if name in df.columns
]
trajectory_rows = []
for checkpoint in CHECKPOINT_ORDER:
    checkpoint_df = df[df["checkpoint"] == checkpoint]
    for feature in signal_family_columns:
        for satd_value, label in [(0, "Non-SATD"), (1, "SATD")]:
            values = pd.to_numeric(
                checkpoint_df.loc[checkpoint_df["is_satd"] == satd_value, feature],
                errors="coerce",
            ).dropna()
            trajectory_rows.append({
                "checkpoint": checkpoint, "feature": feature, "group": label,
                "n": len(values), "mean": values.mean(), "median": values.median(),
                "q1": values.quantile(0.25), "q3": values.quantile(0.75),
            })
trajectory_summary = pd.DataFrame(trajectory_rows)
trajectory_summary.to_csv(OUTPUT_DIR / "signal_trajectory_summary.csv", index=False)
print(trajectory_summary.to_string(index=False))


checkpoint                      feature    group    n     mean  median  q1  q3
        T0      scope_instability_count Non-SATD 7379 0.000000     0.0 0.0 0.0
        T0      scope_instability_count     SATD 2128 0.000000     0.0 0.0 0.0
        T0 estimation_instability_count Non-SATD 7379 0.000000     0.0 0.0 0.0
        T0 estimation_instability_count     SATD 2128 0.000000     0.0 0.0 0.0
        T0  ownership_instability_count Non-SATD 7379 0.000000     0.0 0.0 0.0
        T0  ownership_instability_count     SATD 2128 0.000000     0.0 0.0 0.0
        T0      status_transition_count Non-SATD 7379 0.000000     0.0 0.0 0.0
        T0      status_transition_count     SATD 2128 0.000000     0.0 0.0 0.0
       T25      scope_instability_count Non-SATD 7379 0.006640     0.0 0.0 0.0
       T25      scope_instability_count     SATD 2128 0.007989     0.0 0.0 0.0
       T25 estimation_instability_count Non-SATD 7379 0.005963     0.0 0.0 0.0
       T25 estimation_instability_count     SATD 212

## 3. Aligned snapshots and identical project-aware folds

Sorting each snapshot by the same issue IDs guarantees paired predictions. The
folds are constructed once from T0 and reused unchanged at T25, T50, and T75.


In [6]:
common_ids = sorted(set.intersection(*[
    set(df.loc[df["checkpoint"] == checkpoint, "issue_id"])
    for checkpoint in CHECKPOINT_ORDER
]))
snapshot_dfs = {}
for checkpoint in CHECKPOINT_ORDER:
    snapshot = (
        df[(df["checkpoint"] == checkpoint) & df["issue_id"].isin(common_ids)]
        .copy().set_index("issue_id").loc[common_ids].reset_index()
    )
    snapshot_dfs[checkpoint] = snapshot

reference_labels = snapshot_dfs["T0"]["is_satd"].to_numpy()
reference_projects = snapshot_dfs["T0"]["project_id"].to_numpy()
for checkpoint in CHECKPOINT_ORDER:
    assert np.array_equal(reference_labels, snapshot_dfs[checkpoint]["is_satd"].to_numpy())
    assert np.array_equal(reference_projects, snapshot_dfs[checkpoint]["project_id"].to_numpy())

base = snapshot_dfs["T0"]
y_base = base["is_satd"]
groups_base = base["project_id"]
n_projects = groups_base.nunique()
if n_projects < 3:
    raise ValueError("At least three projects are required for nested grouped validation.")
n_splits = min(5, n_projects)
outer_cv = StratifiedGroupKFold(
    n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE
)
outer_splits = list(outer_cv.split(np.zeros(len(base)), y_base, groups_base))

fold_audit_rows = []
for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    train_projects = set(groups_base.iloc[train_idx])
    test_projects = set(groups_base.iloc[test_idx])
    assert train_projects.isdisjoint(test_projects)
    fold_audit_rows.append({
        "fold": fold, "train_issues": len(train_idx), "test_issues": len(test_idx),
        "train_projects": len(train_projects), "test_projects": len(test_projects),
        "train_satd_prevalence": y_base.iloc[train_idx].mean(),
        "test_satd_prevalence": y_base.iloc[test_idx].mean(),
        "test_project_ids": ",".join(map(str, sorted(test_projects))),
    })
fold_audit = pd.DataFrame(fold_audit_rows)
fold_audit.to_csv(OUTPUT_DIR / "outer_fold_audit.csv", index=False)
print(f"Aligned issues: {len(common_ids):,}; outer folds: {len(outer_splits)}")
print(fold_audit.to_string(index=False))


Aligned issues: 9,507; outer folds: 5
 fold  train_issues  test_issues  train_projects  test_projects  train_satd_prevalence  test_satd_prevalence test_project_ids
    1          8287         1220              20              5               0.232292              0.166393   14,16,20,23,25
    2          6784         2723              20              5               0.240271              0.182887     1,3,11,15,27
    3          8243         1264              20              5               0.212908              0.295095    9,10,17,22,24
    4          8124         1383              21              4               0.232767              0.171367        2,6,19,26
    5          6590         2917              19              6               0.198938              0.280082    4,5,7,8,18,21


## 4. Models and nested threshold selection

Logistic regression provides an interpretable linear baseline; XGBoost captures
nonlinearities and interactions. Threshold-dependent precision, recall, and F1 use
a threshold learned only from out-of-fold predictions inside the outer training
projects. Ranking and probability metrics remain threshold-free.


In [7]:
def build_model_pipeline(model_type="lr"):
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    preprocessor = ColumnTransformer([
        ("categorical", categorical_transformer, planning_categorical),
        ("numeric", numeric_transformer, planning_numeric + dynamic_features),
    ])
    if model_type == "lr":
        model = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
    elif model_type == "xgb":
        model = XGBClassifier(
            n_estimators=500, max_depth=4, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8,
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
        )
    else:
        raise ValueError("model_type must be 'lr' or 'xgb'")
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def select_threshold_inner_cv(X_train, y_train, groups_train, model_type, random_state):
    n_inner = min(4, groups_train.nunique())
    if n_inner < 3 or y_train.nunique() < 2:
        return 0.5
    inner_cv = StratifiedGroupKFold(
        n_splits=n_inner, shuffle=True, random_state=random_state
    )
    probabilities = np.full(len(y_train), np.nan)
    for inner_train_idx, inner_val_idx in inner_cv.split(X_train, y_train, groups_train):
        inner_y = y_train.iloc[inner_train_idx]
        if inner_y.nunique() < 2:
            continue
        model = build_model_pipeline(model_type)
        model.fit(X_train.iloc[inner_train_idx], inner_y)
        probabilities[inner_val_idx] = model.predict_proba(X_train.iloc[inner_val_idx])[:, 1]
    valid = np.isfinite(probabilities)
    if not valid.any():
        return 0.5
    valid_y = y_train.iloc[np.flatnonzero(valid)].to_numpy()
    candidates = np.linspace(0.05, 0.95, 181)
    scores = [
        f1_score(valid_y, (probabilities[valid] >= threshold).astype(int), zero_division=0)
        for threshold in candidates
    ]
    return float(candidates[int(np.argmax(scores))])


def evaluate_checkpoint(snapshot_df, checkpoint, model_type, splits):
    X = snapshot_df[all_features].copy()
    y = snapshot_df["is_satd"].copy()
    groups = snapshot_df["project_id"].copy()
    fold_rows = []
    oof_probability = np.full(len(snapshot_df), np.nan)
    oof_prediction = np.full(len(snapshot_df), np.nan)
    oof_threshold = np.full(len(snapshot_df), np.nan)
    oof_null_probability = np.full(len(snapshot_df), np.nan)
    oof_fold = np.full(len(snapshot_df), np.nan)

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]
        threshold = select_threshold_inner_cv(
            X_train, y_train, groups_train, model_type, RANDOM_STATE + fold
        )
        model = build_model_pipeline(model_type)
        model.fit(X_train, y_train)
        probability = model.predict_proba(X_test)[:, 1]
        prediction = (probability >= threshold).astype(int)
        train_prevalence = y_train.mean()
        null_probability = np.full(len(y_test), train_prevalence)
        prevalence = y_test.mean()
        pr_auc = average_precision_score(y_test, probability)
        roc_auc = roc_auc_score(y_test, probability) if y_test.nunique() == 2 else np.nan
        brier = brier_score_loss(y_test, probability)
        null_brier = brier_score_loss(y_test, null_probability)
        fold_rows.append({
            "checkpoint": checkpoint, "model": model_type, "fold": fold,
            "test_issues": len(test_idx), "test_projects": groups.iloc[test_idx].nunique(),
            "satd_prevalence": prevalence, "threshold": threshold,
            "precision": precision_score(y_test, prediction, zero_division=0),
            "recall": recall_score(y_test, prediction, zero_division=0),
            "f1": f1_score(y_test, prediction, zero_division=0),
            "pr_auc": pr_auc,
            "pr_auc_lift": pr_auc / prevalence if prevalence > 0 else np.nan,
            "roc_auc": roc_auc, "brier": brier, "null_brier": null_brier,
            "brier_skill": 1 - brier / null_brier if null_brier > 0 else np.nan,
        })
        oof_probability[test_idx] = probability
        oof_prediction[test_idx] = prediction
        oof_threshold[test_idx] = threshold
        oof_null_probability[test_idx] = null_probability
        oof_fold[test_idx] = fold

    prediction_df = snapshot_df[
        ["issue_id", "project_id", "sprint_id", "is_satd"]
    ].copy()
    prediction_df["checkpoint"] = checkpoint
    prediction_df["model"] = model_type
    prediction_df["fold"] = oof_fold.astype(int)
    prediction_df["probability"] = oof_probability
    prediction_df["prediction"] = oof_prediction
    prediction_df["threshold"] = oof_threshold
    prediction_df["null_probability"] = oof_null_probability
    if prediction_df["probability"].isna().any():
        raise RuntimeError(f"Missing OOF predictions for {model_type}/{checkpoint}")
    return pd.DataFrame(fold_rows), prediction_df


In [8]:
all_fold_results = []
all_predictions = []
for model_type in ["lr", "xgb"]:
    for checkpoint in CHECKPOINT_ORDER:
        print(f"Running {model_type.upper()} at {checkpoint} ...", flush=True)
        fold_result, prediction = evaluate_checkpoint(
            snapshot_dfs[checkpoint], checkpoint, model_type, outer_splits
        )
        all_fold_results.append(fold_result)
        all_predictions.append(prediction)

fold_results = pd.concat(all_fold_results, ignore_index=True)
predictions = pd.concat(all_predictions, ignore_index=True)
lr_predictions = predictions[predictions["model"] == "lr"].copy()
xgb_predictions = predictions[predictions["model"] == "xgb"].copy()
fold_results.to_csv(OUTPUT_DIR / "checkpoint_fold_results.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "checkpoint_oof_predictions.csv", index=False)
print("Model evaluation complete.")


Running LR at T0 ...
Running LR at T25 ...
Running LR at T50 ...
Running LR at T75 ...
Running XGB at T0 ...
Running XGB at T25 ...
Running XGB at T50 ...
Running XGB at T75 ...
Model evaluation complete.


## 5. Predictive performance and probability reliability

PR-AUC is primary because SATD is the minority class. Its random-ranking baseline
is the SATD prevalence. Brier skill compares each model with the outer-training
prevalence forecast; positive values indicate improved probability forecasts.


In [9]:
flat_summary = (
    fold_results.groupby(["model", "checkpoint"], observed=True)
    .agg(
        precision_mean=("precision", "mean"), precision_std=("precision", "std"),
        recall_mean=("recall", "mean"), recall_std=("recall", "std"),
        f1_mean=("f1", "mean"), f1_std=("f1", "std"),
        pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"),
        pr_auc_lift_mean=("pr_auc_lift", "mean"),
        roc_auc_mean=("roc_auc", "mean"), roc_auc_std=("roc_auc", "std"),
        brier_mean=("brier", "mean"), brier_std=("brier", "std"),
        brier_skill_mean=("brier_skill", "mean"),
        threshold_mean=("threshold", "mean"),
    ).reset_index()
)
flat_summary["checkpoint"] = pd.Categorical(
    flat_summary["checkpoint"], categories=CHECKPOINT_ORDER, ordered=True
)
flat_summary["remaining_sprint_fraction"] = (
    flat_summary["checkpoint"].astype(str).map(REMAINING_FRACTION)
)
flat_summary["remaining_sprint_percent"] = 100 * flat_summary["remaining_sprint_fraction"]
flat_summary = flat_summary.sort_values(["model", "checkpoint"])
flat_summary.to_csv(OUTPUT_DIR / "checkpoint_performance_flat.csv", index=False)
print(flat_summary.to_string(index=False))

plot_specs = [
    ("pr_auc_mean", "PR-AUC", "SATD early-warning performance", "pr_auc_by_checkpoint.png"),
    ("recall_mean", "Recall", "SATD detection recall", "recall_by_checkpoint.png"),
    ("brier_skill_mean", "Brier skill score", "Probability forecast skill", "brier_skill_by_checkpoint.png"),
]
for metric, ylabel, title, filename in plot_specs:
    fig, ax = plt.subplots(figsize=(8, 5))
    for model_type in ["lr", "xgb"]:
        temp = flat_summary[flat_summary["model"] == model_type]
        ax.plot(temp["checkpoint"].astype(str), temp[metric], marker="o", label=model_type.upper())
    if metric == "pr_auc_mean":
        ax.axhline(satd_prevalence, color="black", linestyle="--", label="Random-ranking baseline")
    if metric == "brier_skill_mean":
        ax.axhline(0, color="black", linestyle="--")
    ax.set(xlabel="Sprint checkpoint", ylabel=ylabel, title=title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=300)
    plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
for checkpoint in CHECKPOINT_ORDER:
    pred = xgb_predictions[xgb_predictions["checkpoint"] == checkpoint]
    prob_true, prob_pred = calibration_curve(
        pred["is_satd"], pred["probability"], n_bins=8, strategy="quantile"
    )
    ax.plot(prob_pred, prob_true, marker="o", label=checkpoint)
ax.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect calibration")
ax.set(
    xlabel="Predicted SATD probability", ylabel="Observed SATD rate",
    title="XGBoost SATD risk calibration across checkpoints",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "calibration_by_checkpoint.png", dpi=300)
plt.close(fig)


model checkpoint  precision_mean  precision_std  recall_mean  recall_std  f1_mean   f1_std  pr_auc_mean  pr_auc_std  pr_auc_lift_mean  roc_auc_mean  roc_auc_std  brier_mean  brier_std  brier_skill_mean  threshold_mean  remaining_sprint_fraction  remaining_sprint_percent
   lr         T0        0.219416       0.062902     0.992888    0.012804 0.356119 0.083604     0.253392    0.078171          1.151623      0.530802     0.054091    0.171763   0.030960          0.002359           0.080                       1.00                     100.0
   lr        T25        0.224688       0.060890     0.910437    0.177625 0.355834 0.084272     0.252588    0.081665          1.143817      0.534312     0.048454    0.173068   0.029406         -0.006555           0.096                       0.75                      75.0
   lr        T50        0.219846       0.062775     0.965777    0.052971 0.355226 0.084452     0.250243    0.080014          1.134062      0.527487     0.051575    0.174075   0.029015    

## 6. Operational early-warning utility

Alert-budget metrics ask how many eventual SATD issues are captured when a team can
inspect only the top 10%, 20%, or 30% of predicted risks. Metrics are computed
separately within each held-out outer fold and then summarized, avoiding a global
ranking across probabilities produced by different fitted models. A pooled OOF
table is also exported as a sensitivity view.


In [10]:
def alert_budget_metrics(prediction_df, budget=0.20):
    data = prediction_df.dropna(subset=["probability", "is_satd"]).sort_values(
        "probability", ascending=False
    )
    n = len(data)
    k = max(1, int(np.ceil(n * budget)))
    alerted = data.iloc[:k]
    total_satd = data["is_satd"].sum()
    detected_satd = alerted["is_satd"].sum()
    precision = alerted["is_satd"].mean()
    recall = detected_satd / total_satd if total_satd > 0 else np.nan
    prevalence = data["is_satd"].mean()
    return {
        "budget": budget, "issues_reviewed": k,
        "precision_at_budget": precision, "recall_at_budget": recall,
        "lift_at_budget": precision / prevalence if prevalence > 0 else np.nan,
    }

fold_alert_rows = []
pooled_alert_rows = []
for (model_type, checkpoint), pred in predictions.groupby(["model", "checkpoint"], observed=True):
    for budget in ALERT_BUDGETS:
        pooled = alert_budget_metrics(pred, budget)
        pooled.update({"model": model_type, "checkpoint": checkpoint})
        pooled_alert_rows.append(pooled)
        for fold, fold_pred in pred.groupby("fold"):
            metrics = alert_budget_metrics(fold_pred, budget)
            metrics.update({"model": model_type, "checkpoint": checkpoint, "fold": int(fold)})
            fold_alert_rows.append(metrics)

alert_fold_results = pd.DataFrame(fold_alert_rows)
pooled_alert_results = pd.DataFrame(pooled_alert_rows)
alert_results = (
    alert_fold_results.groupby(["model", "checkpoint", "budget"], observed=True)
    .agg(
        issues_reviewed_mean=("issues_reviewed", "mean"),
        precision_at_budget=("precision_at_budget", "mean"),
        precision_at_budget_std=("precision_at_budget", "std"),
        recall_at_budget=("recall_at_budget", "mean"),
        recall_at_budget_std=("recall_at_budget", "std"),
        lift_at_budget=("lift_at_budget", "mean"),
        lift_at_budget_std=("lift_at_budget", "std"),
    ).reset_index()
)
alert_fold_results.to_csv(OUTPUT_DIR / "alert_budget_fold_results.csv", index=False)
alert_results.to_csv(OUTPUT_DIR / "alert_budget_results.csv", index=False)
pooled_alert_results.to_csv(OUTPUT_DIR / "alert_budget_pooled_oof_sensitivity.csv", index=False)
print(alert_results.to_string(index=False))


def create_risk_deciles(prediction_df):
    data = prediction_df[prediction_df["probability"].notna()].copy()
    data["risk_decile"] = pd.qcut(
        data["probability"].rank(method="first"), q=10, labels=range(1, 11)
    )
    return (
        data.groupby("risk_decile", observed=True)
        .agg(
            issue_count=("issue_id", "count"),
            observed_satd_rate=("is_satd", "mean"),
            mean_predicted_risk=("probability", "mean"),
        ).reset_index()
    )

decile_parts = []
for checkpoint in CHECKPOINT_ORDER:
    temp = create_risk_deciles(xgb_predictions[xgb_predictions["checkpoint"] == checkpoint])
    temp["checkpoint"] = checkpoint
    decile_parts.append(temp)
decile_results = pd.concat(decile_parts, ignore_index=True)
decile_results["observed_satd_percentage"] = 100 * decile_results["observed_satd_rate"]
decile_results.to_csv(OUTPUT_DIR / "risk_deciles_by_checkpoint.csv", index=False)


model checkpoint  budget  issues_reviewed_mean  precision_at_budget  precision_at_budget_std  recall_at_budget  recall_at_budget_std  lift_at_budget  lift_at_budget_std
   lr         T0     0.1                 190.6             0.294004                 0.085017          0.134664              0.012116        1.342905            0.119142
   lr         T0     0.2                 380.6             0.262213                 0.099082          0.235908              0.032754        1.178603            0.163685
   lr         T0     0.3                 570.8             0.242025                 0.096112          0.324521              0.053908        1.080906            0.179039
   lr        T25     0.1                 190.6             0.290479                 0.102013          0.131380              0.014799        1.310055            0.145081
   lr        T25     0.2                 380.6             0.268320                 0.102373          0.240871              0.031571        1.203392       

## 7. Project-cluster uncertainty and paired checkpoint tests

Confidence intervals resample whole projects. Checkpoint differences are paired on
the same issues and use the same project sample, so they address whether later
information improves ranking beyond sampling variation.


In [11]:
def cluster_bootstrap_metrics(prediction_df, n_boot=N_BOOT, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    project_values = prediction_df["project_id"].to_numpy()
    projects = pd.unique(project_values[pd.notna(project_values)])
    project_rows = [np.flatnonzero(project_values == project) for project in projects]
    rows = []
    for _ in range(n_boot):
        sampled_positions = rng.integers(0, len(projects), size=len(projects))
        sample_rows = np.concatenate([project_rows[position] for position in sampled_positions])
        sample = prediction_df.iloc[sample_rows]
        y_true = sample["is_satd"]
        if y_true.nunique() < 2:
            continue
        probability = sample["probability"]
        null_probability = sample["null_probability"]
        brier = brier_score_loss(y_true, probability)
        null_brier = brier_score_loss(y_true, null_probability)
        rows.append({
            "pr_auc": average_precision_score(y_true, probability),
            "roc_auc": roc_auc_score(y_true, probability),
            "brier": brier,
            "brier_skill": 1 - brier / null_brier if null_brier > 0 else np.nan,
        })
    boot = pd.DataFrame(rows)
    result = {"successful_bootstraps": len(boot)}
    for metric in ["pr_auc", "roc_auc", "brier", "brier_skill"]:
        result[f"{metric}_lower"] = boot[metric].quantile(0.025)
        result[f"{metric}_upper"] = boot[metric].quantile(0.975)
    return result

bootstrap_rows = []
for (model_type, checkpoint), pred in predictions.groupby(["model", "checkpoint"], observed=True):
    ci = cluster_bootstrap_metrics(
        pred, n_boot=N_BOOT,
        random_state=RANDOM_STATE + CHECKPOINT_ORDER.index(str(checkpoint)),
    )
    ci.update({"model": model_type, "checkpoint": checkpoint})
    bootstrap_rows.append(ci)
bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_results.to_csv(OUTPUT_DIR / "checkpoint_bootstrap_ci.csv", index=False)
print(bootstrap_results.to_string(index=False))


xgb_wide = (
    xgb_predictions.pivot_table(
        index=["issue_id", "project_id", "is_satd"],
        columns="checkpoint", values="probability",
    ).reset_index()
)
xgb_wide.columns.name = None


def paired_project_bootstrap_pr_auc(
    wide_df, checkpoint_a, checkpoint_b,
    n_boot=N_PAIRED_BOOT, random_state=RANDOM_STATE,
):
    rng = np.random.default_rng(random_state)
    project_values = wide_df["project_id"].to_numpy()
    projects = pd.unique(project_values[pd.notna(project_values)])
    project_rows = [np.flatnonzero(project_values == project) for project in projects]
    differences = []
    for _ in range(n_boot):
        sampled_positions = rng.integers(0, len(projects), size=len(projects))
        sample_rows = np.concatenate([project_rows[position] for position in sampled_positions])
        sample = wide_df.iloc[sample_rows]
        if sample["is_satd"].nunique() < 2:
            continue
        score_a = average_precision_score(sample["is_satd"], sample[checkpoint_a])
        score_b = average_precision_score(sample["is_satd"], sample[checkpoint_b])
        differences.append(score_b - score_a)
    differences = np.asarray(differences)
    return {
        "checkpoint_a": checkpoint_a, "checkpoint_b": checkpoint_b,
        "successful_bootstraps": len(differences),
        "mean_delta_pr_auc": differences.mean(),
        "ci_lower": np.quantile(differences, 0.025),
        "ci_upper": np.quantile(differences, 0.975),
        "probability_improvement": (differences > 0).mean(),
    }

checkpoint_pairs = [
    ("T0", "T25"), ("T25", "T50"), ("T50", "T75"),
    ("T0", "T50"), ("T0", "T75"),
]
paired_results = pd.DataFrame([
    paired_project_bootstrap_pr_auc(
        xgb_wide, a, b, n_boot=N_PAIRED_BOOT,
        random_state=RANDOM_STATE + pair_number,
    )
    for pair_number, (a, b) in enumerate(checkpoint_pairs)
])
paired_results.to_csv(OUTPUT_DIR / "paired_checkpoint_pr_auc_comparison.csv", index=False)
print("\nPaired XGBoost checkpoint comparisons:")
print(paired_results.to_string(index=False))


 successful_bootstraps  pr_auc_lower  pr_auc_upper  roc_auc_lower  roc_auc_upper  brier_lower  brier_upper  brier_skill_lower  brier_skill_upper model checkpoint
                  1000      0.205672      0.325142       0.508123       0.600652     0.148986     0.197595          -0.013431           0.036641    lr         T0
                  1000      0.198048      0.323894       0.511497       0.598588     0.150407     0.200589          -0.031570           0.029486    lr        T25
                  1000      0.200126      0.318671       0.502607       0.594996     0.154813     0.198571          -0.050210           0.026315    lr        T50
                  1000      0.199810      0.315000       0.508687       0.589543     0.154385     0.199128          -0.060143           0.026242    lr        T75
                  1000      0.220527      0.372716       0.542111       0.634948     0.146154     0.194758          -0.010015           0.061358   xgb         T0
                  1000      

## 8. Earliest actionable checkpoint

The operational criterion below was fixed above: review no more than 20% of issues,
capture at least 50% of eventual SATD issues, and achieve at least 2× lift. This is
a transparent example policy, not a universal scientific threshold; a deployment
should replace it with practitioner-agreed costs and capacity.


In [12]:
def find_earliest_actionable_checkpoint(
    alert_summary, model="xgb", budget=ACTIONABLE_BUDGET,
    min_recall=MIN_RECALL, min_lift=MIN_LIFT,
):
    temp = alert_summary[
        (alert_summary["model"] == model) & np.isclose(alert_summary["budget"], budget)
    ].copy()
    temp["checkpoint"] = pd.Categorical(
        temp["checkpoint"], categories=CHECKPOINT_ORDER, ordered=True
    )
    temp = temp.sort_values("checkpoint")
    temp["actionable"] = (
        (temp["recall_at_budget"] >= min_recall) &
        (temp["lift_at_budget"] >= min_lift)
    )
    candidates = temp[temp["actionable"]]
    earliest = None if candidates.empty else str(candidates.iloc[0]["checkpoint"])
    return earliest, temp

earliest_checkpoint, actionability_table = find_earliest_actionable_checkpoint(alert_results)
actionability_table.to_csv(OUTPUT_DIR / "actionability_at_20pct.csv", index=False)
print(f"Earliest checkpoint meeting the predeclared operational rule: {earliest_checkpoint}")
print(actionability_table.to_string(index=False))


Earliest checkpoint meeting the predeclared operational rule: None
model checkpoint  budget  issues_reviewed_mean  precision_at_budget  precision_at_budget_std  recall_at_budget  recall_at_budget_std  lift_at_budget  lift_at_budget_std  actionable
  xgb         T0     0.2                 380.6             0.294234                 0.105766          0.265620              0.031849        1.327043            0.159196       False
  xgb        T25     0.2                 380.6             0.295763                 0.104771          0.266768              0.023729        1.332769            0.118480       False
  xgb        T50     0.2                 380.6             0.301448                 0.101665          0.272762              0.020979        1.362715            0.104736       False
  xgb        T75     0.2                 380.6             0.300368                 0.098572          0.272594              0.025311        1.361905            0.126829       False


## 9. When individual signals become informative

Permutation importance is measured on held-out projects using PR-AUC. Positive-fold
percentage distinguishes stable signals from effects driven by one split. Disable
`RUN_PERMUTATION_IMPORTANCE` only for a quick diagnostic run.


In [13]:
def checkpoint_permutation_importance(snapshot_df, checkpoint, splits):
    X = snapshot_df[all_features]
    y = snapshot_df["is_satd"]
    rows = []
    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        print(f"Permutation importance: {checkpoint}, fold {fold}", flush=True)
        model = build_model_pipeline("xgb")
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        importance = permutation_importance(
            model, X.iloc[test_idx], y.iloc[test_idx],
            scoring="average_precision", n_repeats=5,
            random_state=100 + fold, n_jobs=1,
        )
        for feature, mean_value, std_value in zip(
            all_features, importance.importances_mean, importance.importances_std
        ):
            rows.append({
                "checkpoint": checkpoint, "fold": fold, "feature": feature,
                "pr_auc_drop": mean_value, "pr_auc_drop_std": std_value,
            })
    return pd.DataFrame(rows)

if RUN_PERMUTATION_IMPORTANCE:
    temporal_importance = pd.concat([
        checkpoint_permutation_importance(snapshot_dfs[checkpoint], checkpoint, outer_splits)
        for checkpoint in ["T25", "T50", "T75"]
    ], ignore_index=True)
    temporal_importance_summary = (
        temporal_importance.groupby(["checkpoint", "feature"], observed=True)
        .agg(
            mean_pr_auc_drop=("pr_auc_drop", "mean"),
            std_pr_auc_drop=("pr_auc_drop", "std"),
            positive_fold_pct=("pr_auc_drop", lambda values: 100 * (values > 0).mean()),
        ).reset_index()
    )
    feature_to_family = {feature: "Planning" for feature in planning_categorical + planning_numeric}
    for family, features in dynamic_families_raw.items():
        feature_to_family.update({f"log_{feature}": family for feature in features if feature in dynamic_raw})
    temporal_importance_summary["family"] = temporal_importance_summary["feature"].map(feature_to_family)
    temporal_importance.to_csv(OUTPUT_DIR / "temporal_feature_importance_folds.csv", index=False)
    temporal_importance_summary.to_csv(OUTPUT_DIR / "temporal_feature_importance.csv", index=False)
    top_importance = (
        temporal_importance_summary.sort_values(
            ["checkpoint", "mean_pr_auc_drop"], ascending=[True, False]
        ).groupby("checkpoint", observed=True).head(10)
    )
    print(top_importance.to_string(index=False))
else:
    temporal_importance = pd.DataFrame()
    temporal_importance_summary = pd.DataFrame()
    print("Permutation importance disabled.")


Permutation importance: T25, fold 1
Permutation importance: T25, fold 2
Permutation importance: T25, fold 3
Permutation importance: T25, fold 4
Permutation importance: T25, fold 5
Permutation importance: T50, fold 1
Permutation importance: T50, fold 2
Permutation importance: T50, fold 3
Permutation importance: T50, fold 4
Permutation importance: T50, fold 5
Permutation importance: T75, fold 1
Permutation importance: T75, fold 2
Permutation importance: T75, fold 3
Permutation importance: T75, fold 4
Permutation importance: T75, fold 5
checkpoint                       feature  mean_pr_auc_drop  std_pr_auc_drop  positive_fold_pct        family
       T25                 log_issue_age          0.043741         0.028253              100.0      Planning
       T25 sprint_issue_count_percentile          0.011479         0.009517              100.0      Planning
       T25         project_sp_percentile          0.009402         0.013501               80.0      Planning
       T25     sprint_sp

## 10. Integrated RQ4 result and defensible wording

The final table joins predictive performance, operational utility, uncertainty,
and remaining intervention time. The generated prose says "eventual SATD label,"
preserving the outcome-timing limitation stated at the start.


In [14]:
primary = flat_summary[flat_summary["model"] == "xgb"].copy()
alert_20 = alert_results[
    (alert_results["model"] == "xgb") & np.isclose(alert_results["budget"], ACTIONABLE_BUDGET)
][[
    "checkpoint", "precision_at_budget", "recall_at_budget", "lift_at_budget",
]]
ci_xgb = bootstrap_results[bootstrap_results["model"] == "xgb"][
    ["checkpoint", "pr_auc_lower", "pr_auc_upper", "brier_lower", "brier_upper"]
]
primary = primary.merge(alert_20, on="checkpoint", validate="one_to_one")
primary = primary.merge(ci_xgb, on="checkpoint", validate="one_to_one")
primary = primary[[
    "checkpoint", "remaining_sprint_percent", "pr_auc_mean", "pr_auc_lower",
    "pr_auc_upper", "pr_auc_lift_mean", "roc_auc_mean", "precision_mean",
    "recall_mean", "f1_mean", "brier_mean", "brier_skill_mean",
    "precision_at_budget", "recall_at_budget", "lift_at_budget",
]]
primary.to_csv(OUTPUT_DIR / "rq4_primary_results.csv", index=False)

action_text = (
    "No checkpoint met the predeclared 20% alert-budget rule."
    if earliest_checkpoint is None
    else f"{earliest_checkpoint} was the earliest checkpoint meeting the predeclared "
         f"20% alert-budget rule (mean recall ≥ {MIN_RECALL:.0%}; mean lift ≥ {MIN_LIFT:.1f}×)."
)
adjacent = paired_results[
    paired_results.apply(
        lambda row: (row["checkpoint_a"], row["checkpoint_b"])
        in [("T0", "T25"), ("T25", "T50"), ("T50", "T75")], axis=1
    )
]
comparison_lines = []
for row in adjacent.itertuples(index=False):
    certainty = "clear" if row.ci_lower > 0 else "uncertain"
    comparison_lines.append(
        f"{row.checkpoint_a}→{row.checkpoint_b}: ΔPR-AUC={row.mean_delta_pr_auc:+.3f} "
        f"(95% project-bootstrap CI {row.ci_lower:+.3f} to {row.ci_upper:+.3f}; {certainty})."
    )
narrative = (
    "RQ4 answer (XGBoost, out-of-project validation)\n"
    f"The eventual-SATD prevalence was {satd_prevalence:.1%}. {action_text}\n"
    + "\n".join(comparison_lines)
    + "\nThese results estimate prediction of the eventual text-derived SATD label; they do "
      "not establish that the warning precedes the first appearance of SATD text."
)
(OUTPUT_DIR / "rq4_answer.txt").write_text(narrative, encoding="utf-8")
print(primary.round(4).to_string(index=False))
print("\n" + narrative)
print("\nArtifacts written to", OUTPUT_DIR.resolve())


checkpoint  remaining_sprint_percent  pr_auc_mean  pr_auc_lower  pr_auc_upper  pr_auc_lift_mean  roc_auc_mean  precision_mean  recall_mean  f1_mean  brier_mean  brier_skill_mean  precision_at_budget  recall_at_budget  lift_at_budget
        T0                     100.0       0.2902        0.2205        0.3727            1.3178        0.5816          0.2195       0.9296   0.3518      0.1679            0.0256               0.2942            0.2656          1.3270
       T25                      75.0       0.2883        0.2187        0.3789            1.3061        0.5807          0.2195       0.9352   0.3524      0.1679            0.0254               0.2958            0.2668          1.3328
       T50                      50.0       0.2905        0.2154        0.3769            1.3164        0.5848          0.2202       0.9151   0.3511      0.1677            0.0268               0.3014            0.2728          1.3627
       T75                      25.0       0.2955        0.2233     